In [1]:
import geopandas as gpd
from rasterstats import zonal_stats
import pandas as pd
import rasterio
import os
import numpy as np
from rasterio.mask import mask

In [2]:
base = os.path.join(os.getcwd(),'..')

In [3]:
shapefile_path = os.path.join(os.getcwd(),'..','shape','States_shapefile.shp')

In [4]:
gdf = gpd.read_file(shapefile_path)

In [5]:
outPath = os.path.join(base,'stats')
if not os.path.exists(outPath):
    os.mkdir(outPath)

In [6]:
def get_zonal(fn,bn,ct):
    name = fn.split('\\')[-1].split('.')[0]
    
    with rasterio.open(fn) as src:
        results = []
        for idx, row in gdf.iterrows():
            geom = [row.geometry]  
            name_s = row['State_Name']
            out_image, out_transform = mask(src, geom, crop=True)
            
            band = out_image[bn] 
            
            if src.nodata is not None:
                band = band[band != src.nodata]
            
            band = band[~np.isnan(band)]
            band = np.where(band==0,np.nan,band)
            if band.size > 0:
                stats = {
                    "zone_id": name_s,
                    "mean": band.mean(),
                    "sum": band.sum(),
                    "min": band.min(),
                    "max": band.max(),
                    "count": band.size
                }
            else:
                stats = {
                    "zone_id": name_s,
                    "mean": np.nan,
                    "sum": np.nan,
                    "min": np.nan,
                    "max": np.nan,
                    "count": 0
                }
    
            results.append(stats)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    fo = os.path.join(outPath,name+'_'+ct+'_maize.csv')
    print(fo)
    df.to_csv(fo, index=False)

In [7]:
raster = os.path.join(base,'tiffs')
os.listdir(raster)

['aboveground_biomassc',
 'C_leaf_pft',
 'C_pool_pft',
 'C_so_pft',
 'evap',
 'hdate',
 'LAI',
 'sdate',
 'test.tiff',
 'test.tiff.aux.xml',
 'transp']

In [8]:

agbm = os.path.join(raster,'sdate')

In [10]:
bn=2
ct='rainfed'
for r,d,f in os.walk(agbm):
    for fl in f:
        if fl.endswith('.tif'):
            
            fn = os.path.join(r,fl)
            name = fn.split('\\')[-1].split('.')[0]
            get_zonal(fn,bn,ct)
print('processing completed')

notebooks\..\stats\sdate_time_000_rainfed_maize.csv
notebooks\..\stats\sdate_time_001_rainfed_maize.csv
notebooks\..\stats\sdate_time_002_rainfed_maize.csv
notebooks\..\stats\sdate_time_003_rainfed_maize.csv
notebooks\..\stats\sdate_time_004_rainfed_maize.csv
processing completed
